# 0617 完整報告內容：最終版對比 0609ver_3

## 報告定位

```text
1. 資料進 DB 後，少榆端要能檢查與產生 alert_event。
2. alert_event、future_prediction_result 等結果不能只停在 JSON，要能回寫 Database。
3. past / current / future 要整合成 service，而不是各自分散。
4. UI 不能只看到前端 demo，要能透過 API 呼叫少榆端整理後的內容。
5. 正式 DB 查詢與寫回仍要走余宇承的 Database/versionB。
```


In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd()

# 建議把此 Notebook 放在 少榆0617ver_1 根目錄執行。
# 如果不是在根目錄，會嘗試往下尋找。
if not (PROJECT_DIR / "webservices").exists():
    candidates = list(Path.cwd().glob("**/少榆0617ver_1"))
    if candidates:
        PROJECT_DIR = candidates[0]

print("PROJECT_DIR =", PROJECT_DIR)
print("webservices exists:", (PROJECT_DIR / "webservices").exists())

def show_file(path, start=1, end=None, keyword=None, max_lines=90):
    p = PROJECT_DIR / path
    print("=" * 100)
    print(path)
    print("=" * 100)

    if not p.exists():
        print("找不到檔案：", p)
        return

    text = p.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    if keyword:
        hits = [i for i, line in enumerate(lines, 1) if keyword in line]
        if not hits:
            print("找不到 keyword:", keyword)
            return
        start = max(1, hits[0] - 8)
        end = min(len(lines), hits[0] + max_lines)

    if end is None:
        end = min(len(lines), start + max_lines)

    for i, line in enumerate(lines[start-1:end], start):
        print(f"{i:04d}: {line}")

def check_exists(paths):
    for path in paths:
        print(f"{'OK' if (PROJECT_DIR / path).exists() else 'MISS'}  {path}")

def show_json(path, max_chars=2600):
    p = PROJECT_DIR / path
    if not p.exists():
        print("找不到檔案：", p)
        return
    data = json.loads(p.read_text(encoding="utf-8", errors="replace"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars])
    if len(text) > max_chars:
        print("\n...省略，報告時可打開完整檔案")


# 一、0609ver_3 當時的狀態與老師提醒的問題

## 主旨

`0609ver_3` 當時比較接近「資料格式規劃版」。  
當時少榆端已經有 schema、CSV template、API requirement、原因對策矩陣等設計資料，但整體流程還沒有真的串起來。

老師上次主要不是要看更多 template，而是要確認系統流程：

```text
資料進 DB 後，誰定期檢查？
異常事件由誰產生？
alert_event 要寫到哪裡？
future prediction 要怎麼讓 UI 和 DB 都看得到？
UI 要怎麼取得少榆端產出的結果？
```

所以這次報告不再著重中間版本做了哪些小修改，而是直接說明：  
**0617ver_1 如何把 0609ver_3 缺的「資料流、API、DB 回寫、UI 串接」補完整。**

## 相關檔案

這一節主要是問題定位，不一定需要開程式檔。  
若老師要看最終對應檔案，可以從下列檔案開始：

```text
webservices/api_server.py
webservices/time_series_api/src/api_server.py
webservices/integrated_service/sprayline_integrated_service.py
webservices/integration_adapter/database_versionb_adapter.py
```

## 做法

四個最終任務：

### 任務一：DB 端要看得到產生的結果

不是只產生 JSON 或 template，而是把 alert、status、future prediction 寫回正式 DB。

### 任務二：UI 端要能呼叫功能

過去少榆端比較像內部 Python function，前端 UI 不能直接抓。  
0617ver_1 補上 FastAPI，讓 UI 可以透過 HTTP API 呼叫。

### 任務三：Past / Current / Future 要整合

past/current 不能只在 UI demo 裡，future 也不能只在少榆端各自處理。  
0617ver_1 將 past/current/future 統一整理成可查詢、可顯示、可接 DB 回寫的流程。

### 任務四：正式 DB 流程不能被 demo JSON 取代

TimeSeriesService 可以提供 UI 顯示格式，但正式查詢與寫回仍要走余宇承 `Database/versionB`。


# 二、0617ver_1 最終完成的整體架構

## 主旨

`0617ver_1` 最重要的改動，是把少榆端整理成一個完整的「前後端都能使用」的 service 架構。  
也就是說，現在不再只是 DB 或 Python 後端能看見少榆 function，而是 UI 也能透過 API 呼叫少榆端整理後的內容。

最終架構如下：

```text
UI
-> FastAPI API Server
-> TimeSeries UI 查詢線 / Service Orchestration 正式整合線
-> IntegratedSprayLineService / FutureService / MonitoringWorker / TroubleshootingService
-> Database/versionB
-> PostgreSQL
```

## 相關檔案

```text
webservices/api_server.py
webservices/time_series_api/src/api_server.py
webservices/time_series_api/src/shaoyu_ui_bridge.py
webservices/time_series_api/src/service_orchestration_adapter.py
webservices/integrated_service/sprayline_integrated_service.py
webservices/integration_adapter/database_versionb_adapter.py
scripts/run_api_server.py
scripts/run_api_smoke_test.py
```

## 做法

0617ver_1 的做法不是把所有程式混在一起，而是把不同責任分層：

### 第一層：API 對外入口

由 `webservices/api_server.py` 提供統一入口。  
UI 不需要知道少榆內部有哪些 Python class，只要呼叫 API endpoint。

### 第二層：兩條 API 線

API 內分成兩條線：

```text
TimeSeries UI 查詢線：給 UI 畫 past/current/future 畫面
Service Orchestration 線：負責少榆正式功能與 DB 回寫
```

### 第三層：少榆端核心 service

少榆端維持原本的功能分工：

```text
IntegratedSprayLineService：past/current/future 整合
MonitoringWorker：異常判斷與 alert_event
FutureService：future_prediction_result
TroubleshootingService：原因對策
```

### 第四層：Database/versionB

正式查詢與寫回仍走余宇承 `Database/versionB`，不把 demo JSON 當成正式 persistence。


In [ ]:
check_exists([
    "webservices/api_server.py",
    "webservices/time_series_api/src/api_server.py",
    "webservices/time_series_api/src/shaoyu_ui_bridge.py",
    "webservices/time_series_api/src/service_orchestration_adapter.py",
    "webservices/integrated_service/sprayline_integrated_service.py",
    "webservices/integration_adapter/database_versionb_adapter.py",
    "scripts/run_api_server.py",
    "scripts/run_api_smoke_test.py",
])


# 三、API 如何讓 UI 直接呼叫少榆端功能

## 主旨

在 0609ver_3 時，少榆端主要是 function、schema、template。  
這代表後端或 DB 組可以理解少榆端想做什麼，但前端 UI 沒有一個可以直接呼叫的入口。

0617ver_1 補上 FastAPI 後，UI 可以透過 HTTP API 取得少榆端內容。  
這是相對於 0609ver_3 的重要差異：

```text
過去：DB 或 Python 後端才比較能看見少榆 function
現在：UI 可以透過 API 直接取得少榆整理後的資料
```

## 相關檔案

```text
webservices/api_server.py
webservices/time_series_api/src/api_server.py
scripts/run_api_server.py
docs/contracts/0617ver_1_UI_API對照表.md
```

## 做法

### 1. 提供統一 API Server

啟動 API：

```powershell
python scripts\run_api_server.py
```

或：

```powershell
uvicorn webservices.api_server:app --host 0.0.0.0 --port 8001 --reload
```

打開：

```text
http://127.0.0.1:8001/docs
```

UI 組可以直接從 Swagger 查看 endpoint 與 request/response 格式。

### 2. UI 查詢 API 和正式寫回 API 分開

UI 平常拖 slider、看 summary、看 station detail 時，只需要查詢，不應該每次都寫 DB。  
因此 0617ver_1 把 API 分成查詢線與正式回寫線。

### 3. 保留後端可用性

雖然現在有 HTTP API，但少榆端內部仍保留 Python service function。  
這樣 UI 可以透過 API 呼叫，後端測試或 DB 串接也可以直接呼叫 Python function。


In [ ]:
show_file("webservices/api_server.py", start=1, end=90)
print("\n" + "=" * 100 + "\n")
show_file("scripts/run_api_server.py", start=1, end=120)


# 四、兩條 API 線的最終整合

## 主旨

0617ver_1 不是只有新增 API server，而是把前面討論的兩條線整理清楚：

```text
第一條：TimeSeriesService 的 past/current/future UI 查詢線
第二條：Service Orchestration 的少榆正式整合線
```

這兩條線都放在同一個 FastAPI server 裡，但用途不同。  
這樣 UI 組可以依照需求選擇正確 endpoint，而不是每個功能都分散在不同 service 裡。

## 相關檔案

```text
webservices/time_series_api/src/api_server.py
webservices/time_series_api/src/shaoyu_ui_bridge.py
webservices/time_series_api/src/service_orchestration_adapter.py
```

## 做法

### 第一條線：TimeSeriesService UI 查詢線

主要給 UI 顯示用，包含：

```text
time slider
past/current/future time-series
summary
station detail
component detail
```

主要 API：

```text
POST /api/time-series
POST /api/time-series/ui/summary
POST /api/time-series/ui/station-detail
POST /api/time-series/ui/component-detail
```

### 第二條線：Service Orchestration 正式整合線

主要負責少榆端正式 service 流程，包含：

```text
IntegratedSprayLineService
FutureService
MonitoringWorker
TroubleshootingService
Database/versionB
```

主要 API：

```text
POST /api/service-orchestration/integrated/query
POST /api/service-orchestration/integrated/run-once
POST /api/service-orchestration/future/payload
POST /api/service-orchestration/future/save
POST /api/service-orchestration/monitoring/run
GET  /api/service-orchestration/troubleshooting/matrix
```

### 報告重點

```text
TimeSeries 線負責 UI 查詢與顯示格式。
Service Orchestration 線負責少榆正式邏輯與 DB 回寫。
兩者都透過同一個 API server 提供給 UI 或後端使用。
```


In [ ]:
show_file("webservices/time_series_api/src/api_server.py", keyword='@app.post("/api/time-series"')
print("\n" + "=" * 100 + "\n")
show_file("webservices/time_series_api/src/api_server.py", keyword='@app.post("/api/service-orchestration/integrated/query"')


# 五、Past / Current / Future 如何整合

## 主旨

老師上次提到不能只看單一時間點，也要能思考資料流和 UI 如何取得資料。  
0617ver_1 將 past/current/future 整合在 `IntegratedSprayLineService` 中，讓 UI 查詢和少榆後端邏輯可以使用同一套時間資料概念。

整合重點包含：

```text
past / current 資料取得
time slider 概念
current snapshot
past window
UI 需要的 time-series format
future prediction payload
```

## 相關檔案

```text
webservices/integrated_service/sprayline_integrated_service.py
webservices/time_series_api/src/shaoyu_ui_bridge.py
schema/time_series_response.schema.json
schema/sensor_snapshot.schema.json
schema/sensor_feature_window.schema.json
```

## 做法

### 1. Time slider 概念

用 `slider_value` 判斷使用者要看的時間：

```text
slider_value < 0  -> past
slider_value == 0 -> current
slider_value > 0  -> future
```

### 2. Past window

past 模式不是只抓單一點，而是抓一段過去時間窗。  
這樣 UI 可以畫趨勢線，也可以計算歷史摘要。

### 3. Current snapshot

current 模式會抓最近 window 內的資料，整理成目前狀態。  
這份資料可以提供 UI 顯示，也可以作為 Monitoring 或 Future 的輸入。

### 4. Future prediction

future 模式使用 current snapshot 或 current window 作為 input，產生 `future_prediction_payload`。  
如果需要正式保存，再透過 `future/save` 或 `write_back=true` 寫入 DB。

### 5. UI time-series format

最後將資料整理成 UI 可用格式，例如：

```text
viewer_state
stations
time_series.points
past_window.summary
current_snapshot
future_prediction_payload
```


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def build_time_window")
print("\n" + "=" * 100 + "\n")
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def build_ui_time_series_response")


# 六、Alert / Status / Future 如何回寫 Database/versionB

## 主旨

老師上次逐字稿中很重要的一點是：  
少榆端產生的結果不能只停在前端畫面或 JSON 檔，而是要能回寫 DB。

0617ver_1 保留 0616ver_4 已經建立的正式 DB 回寫主線。  
最終可以寫回：

```text
alert_event
alert_cause_link
alert_response_link
batch_station_status
future_prediction_result
```

## 相關檔案

```text
webservices/monitoring_worker/monitoring_worker.py
webservices/monitoring_worker/alert_event_writer.py
webservices/monitoring_worker/batch_station_status_writer.py
webservices/future_service/future_service.py
webservices/integration_adapter/database_versionb_adapter.py
```

## 做法

### 1. Alert event 回寫

`MonitoringWorker` 讀取 sensor 資料後，依照 threshold 判斷 warning / fault。  
若偵測到異常，會由 `alert_event_writer.py` 寫入 `alert_event`。

### 2. 原因與對策 link

異常不只寫入文字，而是透過 cause_id / response_id 對應：

```text
alert_cause_link
alert_response_link
```

這樣 Engineer UI 後續可以查到原因與建議處理方式。

### 3. Batch station status 更新

`batch_station_status_writer.py` 會更新每個 station / component 的狀態，例如 nozzle、filter、compressor、quality 等。

### 4. Future prediction 寫回

`FutureService` 產生 `future_prediction_result` payload 後，透過 `db_future.insert_future_prediction_result()` 寫入正式 DB。

### 5. DB function 不重寫 SQL

所有正式 DB 動作都透過 `Database/versionB` function。  
少榆端不自行亂寫 SQL，避免和余宇承 DB schema 不一致。


In [ ]:
show_file("webservices/monitoring_worker/alert_event_writer.py", start=1, end=130)
print("\n" + "=" * 100 + "\n")
show_file("webservices/monitoring_worker/batch_station_status_writer.py", start=1, end=130)
print("\n" + "=" * 100 + "\n")
show_file("webservices/future_service/future_service.py", start=1, end=150)


# 七、UI 查詢和 DB 寫回分開，避免前端誤寫資料

## 主旨

UI 拖 slider 或查看畫面時，不應該每次都觸發 DB 寫入。  
如果沒有區分查詢與寫回，前端每操作一次，就可能產生大量重複的 alert_event 或 future_prediction_result。

所以 0617ver_1 將 API 分成兩類：

```text
查詢 API：給 UI 看資料，不寫 DB
寫回 API：明確要產生事件或保存預測結果時才使用
```

## 相關檔案

```text
webservices/time_series_api/src/api_server.py
webservices/time_series_api/src/service_orchestration_adapter.py
webservices/integrated_service/sprayline_integrated_service.py
```

## 做法

### UI 查詢 API

適合 UI 拖 slider 或顯示畫面：

```text
POST /api/time-series
POST /api/time-series/ui/summary
POST /api/time-series/ui/station-detail
POST /api/time-series/ui/component-detail
POST /api/service-orchestration/integrated/query
```

這些 API 預設只查詢，不應寫 DB。

### DB 寫回 API

適合明確觸發後端流程：

```text
POST /api/service-orchestration/future/save
POST /api/service-orchestration/monitoring/run
POST /api/service-orchestration/integrated/run-once
```

其中 `integrated/run-once` 只有在 `write_back=true` 時才寫 DB。

### 報告重點

```text
這樣 UI 可以正常查資料，DB 也不會因為 slider 操作而被大量重複寫入。
```


# 八、Database/versionB 連接與 versionB loader 修正

## 主旨

0617ver_1 需要同時支援 API 與 DB，因此必須確保 API 找到的是正式 `Database/versionB`，不是備用 fallback。  
如果路徑找錯，會出現「API 看起來有跑，但其實不是接正式 DB function」的問題。

## 相關檔案

```text
webservices/time_series_api/src/versionb_loader.py
webservices/integration_adapter/database_versionb_adapter.py
```

## 做法

### versionB loader 搜尋順序

0617ver_1 將正式 DB 路徑放在前面：

```text
1. VERSIONB_PATH / SPRAYLINE_DB_FUNCTION_PATH 環境變數
2. SPRAYLINE_PROJECT_ROOT/Database/versionB
3. Project-SprayLine-main/Database/versionB
4. 少榆資料夾 external/Database/versionB
5. time_series_api/external/versionB fallback
```

### 報告時要提醒

實測時應該檢查：

```text
GET /api/versionb/status
```

確認 path 指到：

```text
Project-SprayLine-main/Database/versionB
```

而不是：

```text
webservices/time_series_api/external/versionB
```

### DB 實測限制

目前本機可以測 API 架構、route、fallback 是否正常。  
PostgreSQL 端到端實測仍需在有 DB 的電腦上進行。


In [ ]:
show_file("webservices/time_series_api/src/versionb_loader.py", start=1, end=150)


# 九、目前可展示的檔案與測試方式

## 主旨

正式報告時，不需要把所有檔案都打開。  
建議只打開能證明「API 可被 UI 呼叫、past/current/future 已整合、DB 回寫流程仍保留」的檔案。

## 建議展示檔案

```text
webservices/api_server.py
webservices/time_series_api/src/api_server.py
webservices/time_series_api/src/shaoyu_ui_bridge.py
webservices/time_series_api/src/service_orchestration_adapter.py
webservices/integrated_service/sprayline_integrated_service.py
webservices/integration_adapter/database_versionb_adapter.py
scripts/run_api_server.py
scripts/run_api_smoke_test.py
docs/contracts/0617ver_1_UI_API對照表.md
0617ver_1_PostgreSQL_DB_API完整串接教學.md
```

## 做法

### 1. 啟動 API

```powershell
python scripts\run_api_server.py
```

打開：

```text
http://127.0.0.1:8001/docs
```

### 2. 檢查 route

```powershell
python scripts\run_api_smoke_test.py
```

### 3. 檢查 DB 連線

```powershell
python scripts\check_db_connection.py
```

### 4. 實際 DB 回寫

```powershell
python scripts\run_db_smoke_test.py --write-test-data --station Station_1
```

### 5. SQL 查驗

檢查：

```text
alert_event
alert_cause_link
alert_response_link
batch_station_status
future_prediction_result
```


In [ ]:
check_exists([
    "webservices/api_server.py",
    "webservices/time_series_api/src/api_server.py",
    "webservices/time_series_api/src/shaoyu_ui_bridge.py",
    "webservices/time_series_api/src/service_orchestration_adapter.py",
    "webservices/integrated_service/sprayline_integrated_service.py",
    "webservices/integration_adapter/database_versionb_adapter.py",
    "scripts/run_api_server.py",
    "scripts/run_api_smoke_test.py",
    "docs/contracts/0617ver_1_UI_API對照表.md",
    "0617ver_1_PostgreSQL_DB_API完整串接教學.md",
])


# 十、最後總結稿

## 主旨

0617ver_1 的重點不是中間版本改了多少次，而是針對老師上次指出的問題，整理出最終可串接的 SprayLine API / DB / UI 流程。

## 做法

可以這樣跟老師說：

```text
老師，上次 0609ver_3 時，我這邊主要還是 schema、template 和需求文件。
當時雖然有規劃資料格式，但前端還不能直接呼叫我的內容，
DB 也還沒有完整看到 alert_event、batch_station_status 和 future_prediction_result 的回寫流程。

現在 0617ver_1 已經整理成完整架構：

第一，UI 可以透過 FastAPI 呼叫我的 API。
第二，API 內有 TimeSeriesService 的 past/current/future UI 查詢線。
第三，也有 Service Orchestration 線，負責呼叫我的 IntegratedSprayLineService、MonitoringWorker、FutureService 和 TroubleshootingService。
第四，正式 DB 查詢與寫回仍走余宇承的 Database/versionB。
第五，UI 查詢和 DB 寫回有分開，避免拖 slider 就一直寫 DB。

所以現在前端 UI 可以接我的 API，
後端 DB 也可以看到我產生的 alert_event、batch_station_status 和 future_prediction_result。
```

## 目前仍需實測的部分

```text
1. PostgreSQL 端到端實測要到有 DB 的電腦上執行。
2. UI 前端實際畫面是否完全接上，需要 UI 組測。
3. 若老師要求 batch_summary event 主動寫回 DB，需要再確認 schema。
```

## 最後一句話

```text
0617ver_1 是從 0609ver_3 的規劃版，推進到 UI 可呼叫、後端可串 DB、結果可回寫的 SprayLine API 整合版。
```
